Name: Fahmi Sajid Ahmed / Huzaifa Nissare-Houssen<br>
Student Number: 300250180 / 300172186<br>

<h3><b>Work Split</h3>
<p>
    The assignment was split into 2 distinct parts (Regression Empirical Study with Linear Regression and Classification Empirical Study with Decision Trees). The first of the two (regression) was done solely by Fahmi Sajid Ahmed (300250180). The second of the two (classification) was done by Huzaifa Nissare-Houssen (300172186). Each member did their part in its entirety.
</p>

<h3>Dataset Description</h3>
<p>
    The Vehicle dataset (compiled by Nehal Birla, Nishant Verma, Nikhil Kushwaha) was created for a variety of purposes (i.e. exemplifying the use of linear regression in machine learning). In the case of this assignment, it will be used to perform an empirical study in which we evaluate a linear regression approach on a regression task.
</p>
<p>
    We plan on using the "car details v3.csv" file for the regression analysis. This dataset consists of <b>13 attributes</b> and <b>something rows</b>.
</p>
<p>
    The attributes are as follows:
    <ul>
        <li>name: A categorical attribute defining the name of the vehicle (model)</li>
        <li>year: A numerical attribute defining the year that the car was bought</li>
        <li>selling_price: A numerical attribute defining the selling price of the car (assuming in USD)</li>
        <li>km_driven: A numerical attrribute denoting the number of kilometers that the car was driven</li>
        <li>fuel: A categorical attribute denoting the type of fuel the car uses</li>
        <li>seller_type: A categorical attribute describing if the car was sold by an individual or a dealership</li>
        <li>transmission: A categorical attribute describing if the car is a manual or an automatic</li>
        <li>owner: A categorical attribute denoting the numnber of previous owners the vehicle has had</li>
        <li>mileage: A numerical attribute denoting the mileage of the car</li>
        <li>engine: A numerical attribute describing the engine capacity of the car (in CCs)</li>
        <li>max_power: A numerical attribute denoting the max power of the engine (in bhp)</li>
        <li>torque: A numerical(?) attribute denoting the torque of the car (kgm @ rpm)</li>
        <li>seats: A numerical variable denoting the number of seats in the car</li>
    </ul>
</p>

In [ ]:
# Data Cleaning and Imputation
import pandas as pd;
import numpy as np;
import matplotlib.pyplot as plt;
import seaborn as sns;
import re;
import warnings;
warnings.filterwarnings("ignore");

# Display options for panda
pd.set_option("display.max_columns", None);
pd.set_option("display.float_format", "{:.2f}".format);

# Load the dataset
df = pd.read_csv("Car_details_v3.csv");

# Overview of data (datatypes, missing values, unique values)
summary = pd.DataFrame({
    "dtype":    df.dtypes,
    "missing":  df.isnull().sum(),
    "miss_%":   (df.isnull().mean() * 100).round(2),
    "unique":   df.nunique(),
    "sample":   [df[c].dropna().iloc[0] if df[c].notna().any() else "—" for c in df.columns],
});
print(summary.to_string());

# Check 1: Duplicates
n_dupes = df.duplicated().sum();
print("\nSample duplicated entries:");
print(df[df.duplicated(keep=False)].sort_values("name").head(6));
df.drop_duplicates(inplace=True);
df.reset_index(drop=True, inplace=True);
print(f"\nAfter deduplication: {df.shape[0]:,} rows remain.");

# Check 2: Year Validity
current_year = 2026;
invalid_year = df[(df["year"] < 1885) | (df["year"] > current_year)];
print(f"Rows with year outside [1885, {current_year}]: {len(invalid_year)}");
if not invalid_year.empty:
    print(invalid_year[["name", "year"]].drop_duplicates());

# Check 3: Negative / Zero prices + km_driven
km_thresh = 1_000_000; # extreme threshold
df["km_driven_flag"] = df["km_driven"] > km_thresh;
print(f"\nRows flagged (km_driven > {km_thresh:,}):");
print(df[df["km_driven_flag"]][["name", "year", "km_driven"]]);

# Check 4: Categorical Column Validity
cats = {
    "fuel":         ["Petrol", "Diesel", "CNG", "LPG"],
    "seller_type":  ["Individual", "Dealer", "Trustmark Dealer"],
    "transmission": ["Manual", "Automatic"],
    "owner":        ["First Owner", "Second Owner", "Third Owner",
                     "Fourth & Above Owner", "Test Drive Car"],
};
for col, valid in cats.items():
    bad = df[~df[col].isin(valid)][col].unique();
    flag = "Error" if len(bad) > 0 else "Success";
    print(f"  {flag}  '{col}' — unexpected values: {bad if len(bad) else 'none'}");

# Check 5: seats Validity
suspect_seats = df[df["seats"] > 10]; # More than 10 seats is implausible
print(f"\nRows with seats > 10: {len(suspect_seats)}");
if not suspect_seats.empty:
    print(suspect_seats[["name", "seats"]]);
df.loc[df["seats"] > 10, "seats"] = np.nan; # Replace excess seat values with NaN for imputation

# Some data is represented as a messy string, so the following code section is dedicated to parsing that data into a digestible format
# Parse 1: Mileage (kmpl / km/kg => float)
def parse_mileage(val):
    """Return (numeric_value, unit) or (NaN, None)."""
    if pd.isna(val):
        return np.nan, None;
    m = re.match(r"([\d.]+)\s*(kmpl|km/kg)", str(val).strip(), re.IGNORECASE);
    if m:
        return float(m.group(1)), m.group(2).lower();
    return np.nan, None;
 
df[["mileage_val", "mileage_unit"]] = pd.DataFrame(
    df["mileage"].apply(lambda x: parse_mileage(x)).tolist(), # Apply lambda function to parse mileage function to each
    index=df.index,
);

print(f"Unit counts:\n{df['mileage_unit'].value_counts(dropna=False).to_string()}");
print(f"Unparseable rows: {df['mileage_val'].isna().sum()}");
print(f"mileage_val describe:\n{df['mileage_val'].describe().to_string()}");

# Parse 2: Engine (CC => int)
def parse_engine(val):
    if pd.isna(val):
        return np.nan;
    m = re.match(r"([\d.]+)\s*CC", str(val).strip(), re.IGNORECASE);
    return float(m.group(1)) if m else np.nan;

df["engine_cc"] = df["engine"].apply(parse_engine);
print(f"Unparseable: {df['engine_cc'].isna().sum()}");
print(df["engine_cc"].describe().to_string());

# Parse 3: max power (bhp => float)
def parse_power(val):
    if pd.isna(val):
        return np.nan;
    # Edge-case: some entries are just "bhp" with no associated scalar value
    m = re.match(r"([\d.]+)\s*bhp", str(val).strip(), re.IGNORECASE);
    return float(m.group(1)) if m else np.nan;
 
df["max_power_bhp"] = df["max_power"].apply(parse_power);
print(f"Unparseable: {df['max_power_bhp'].isna().sum()}");
print(df["max_power_bhp"].describe().to_string());

# Parse 4: torque (normalise to Nm)
KGM_TO_NM = 9.80665
 
def parse_torque(val):
    if pd.isna(val):
        return np.nan;
    s = str(val).strip().replace(",", "");
    # Pattern: leading number, optional unit
    m = re.match(r"([\d.]+)\s*(Nm|nm|kgm|KGM)?", s, re.IGNORECASE);
    if not m:
        return np.nan;
    value = float(m.group(1));
    unit  = (m.group(2) or "").lower();
    # If no explicit unit, infer from context
    if unit in ("kgm",):
        value *= KGM_TO_NM;
    elif unit == "" and "kgm" in s.lower():
        value *= KGM_TO_NM;
    return round(value, 2);
 
df["torque_nm"] = df["torque"].apply(parse_torque);
print(f"Unparseable: {df['torque_nm'].isna().sum()}");
print(df["torque_nm"].describe().to_string());

# Check 6: Numeric Range Checks on Parsed Values
checks = {
    "mileage_val  (0–80)":   (df["mileage_val"],    0, 80),
    "engine_cc    (50–6000)":(df["engine_cc"],      50, 6000),
    "max_power_bhp (5–600)": (df["max_power_bhp"],   5, 600),
    "torque_nm   (5–2000)":  (df["torque_nm"],       5, 2000),
    "seats       (2–10)":    (df["seats"],            2, 10),
}
for label, (col, lo, hi) in checks.items():
    out = ((col < lo) | (col > hi)).sum();
    flag = "Error" if out else "Success";
    print(f"  {flag}  {label}: {out} out-of-range rows");
 
# Extreme Mileage Rows
print("\nRows with mileage_val > 50 kmpl (potential data entry errors):");
print(df[df["mileage_val"] > 50][["name", "year", "fuel", "mileage_val"]].head(10));

# Check 7: Co-occurrence of missing values
miss_mask = df[numeric_cols].isnull();
print(miss_mask.sum(axis=1).value_counts().sort_index()
      .rename("rows_with_N_missing_cols").to_string());

# Rows where ALL five parsed columns are missing
all_missing = miss_mask.all(axis=1);
print(f"\nRows missing ALL 5 parsed numeric cols: {all_missing.sum()}");
print(df[all_missing][["name", "year", "fuel", "selling_price"]].head());

# Helper Columns (for grouping)
df["make"] = df["name"].str.split().str[0];
df["year_bin"] = pd.cut(df["year"], bins=range(1980, 2026, 5),
                        labels=[f"{y}–{y+4}" for y in range(1980, 2021, 5)],
                        right=False);

# Imputation 1: Seats (Default Value Imputation (Mode))
def group_mode(series):
    """Return the mode of a series, or NaN if all missing."""
    m = series.dropna().mode();
    return m.iloc[0] if not m.empty else np.nan;
 
before = df["seats"].isna().sum();
df["seats"] = df.groupby(["make", "fuel"], observed=True)["seats"].transform(
    lambda s: s.fillna(group_mode(s))
);
# Fallback (global mode)
global_seats_mode = group_mode(df["seats"]);
df["seats"].fillna(global_seats_mode, inplace=True);
df["seats"] = df["seats"].astype("Int64");
 
print(f"[IMPUTE] seats: {before} NaN => {df['seats'].isna().sum()} NaN remaining");
print(f"Global mode used as fallback: {global_seats_mode}");

# Impute 2: mileage_val, engine_cc, max_power_bhp, torque_nm (Default Value Imputation (Median))
def fill_group_median(df, col, groups):
    before = df[col].isna().sum();
    df[col] = df.groupby(groups, observed=True)[col].transform(
        lambda s: s.fillna(s.median())
    );
    # Fallback (global median)
    global_med = df[col].median();
    df[col].fillna(global_med, inplace=True);
    after = df[col].isna().sum();
    print(f"[IMPUTE] {col:<16}: {before} NaN → {after} NaN  "
          f"(global median fallback = {global_med:.2f})")
    return df;
 
group_cols = ["fuel", "year_bin"];
for col in ["mileage_val", "engine_cc", "max_power_bhp", "torque_nm"]:
    df = fill_group_median(df, col, group_cols);

something

In [ ]:
# Columns to one-hot encode
OHE_COLS = ["fuel", "seller_type", "transmission", "owner"];

# Columns to carry forward as-is (numeric, already clean)
NUMERIC_COLS = [
    "year", "selling_price", "km_driven",
    "mileage_val", "engine_cc", "max_power_bhp", "torque_nm", "seats",
];

# Columns intentionally excluded
EXCLUDED = {
    "name": "High cardinality (2 058 unique values) — OHE would create thousands of sparse columns",
    "make": "Derived helper column; redundant once 'name' is dropped",
    "mileage": "Raw string — superseded by parsed 'mileage_val' / 'mileage_unit'",
    "mileage_unit": "Unit label only; captured implicitly by mileage_val scale",
    "engine": "Raw string — superseded by parsed 'engine_cc'",
    "max_power": "Raw string — superseded by parsed 'max_power_bhp'",
    "torque": "Raw string — superseded by parsed 'torque_nm'",
    "year_bin": "Grouping helper used during imputation only",
    "km_driven_flag": "Boolean audit flag, not a model feature",
};

print("Columns excluded from the modelling DataFrame:");
for col, reason in EXCLUDED.items():
    print(f"  • {col:<16} — {reason}");

# drop_first=True drops one dummy per group to avoid perfect multicollinearity (dummy variable trap), which would make the design matrix singular.
df_encoded = pd.get_dummies(
    df[NUMERIC_COLS + OHE_COLS],
    columns=OHE_COLS,
    drop_first=True,
    dtype=int,  # Encode as 0/1 integers rather than booleans
);

# Identify which dummy was dropped per categorical column (the reference level)
ref_levels = {};
for col in OHE_COLS:
    all_vals   = sorted(df[col].dropna().unique());
    ref_levels[col] = all_vals[0];  # pd.get_dummies drops the first sorted category

print("\nDropped reference levels (drop_first=True):");
for col, ref in ref_levels.items():
    print(f"  - {col:<14} → '{ref}' dropped");

# New dummy columns that were added
original_cols = set(NUMERIC_COLS);
new_dummy_cols = [c for c in df_encoded.columns if c not in original_cols];
print(f"\nDummy columns created ({len(new_dummy_cols)} total):");
for c in new_dummy_cols:
    print(f"  • {c}");

print(f"\nFinal encoded DataFrame: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns");
print(f"Numeric columns: {len(NUMERIC_COLS)}");
print(f"Dummy columns: {len(new_dummy_cols)}");
print(f"Missing values: {df_encoded.isnull().sum().sum()}");

print("\nSample of encoded DataFrame (first 3 rows):");
print(df_encoded.head(3).to_string());

something again

In [ ]:
from sklearn.neighbors import LocalOutlierFactor;
from sklearn.preprocessing import RobustScaler;

# Box plots and Histograms for all numerical features
NUMERIC_FEATURES = [
    "selling_price", "km_driven", "mileage_val",
    "engine_cc", "max_power_bhp", "torque_nm",
];

fig, axes = plt.subplots(2, len(NUMERIC_FEATURES), figsize=(22, 8),
                          gridspec_kw={"hspace": 0.45, "wspace": 0.35});

for i, col in enumerate(NUMERIC_FEATURES):
    data = df[col].dropna();

    # Row 0 — box-plot
    ax_box = axes[0, i];
    ax_box.boxplot(data, vert=True, patch_artist=True,
                   boxprops=dict(facecolor="#AEC6CF", color="#333"),
                   medianprops=dict(color="#C0392B", linewidth=2),
                   flierprops=dict(marker="o", markerfacecolor="#E74C3C",
                                   markersize=3, alpha=0.5));
    ax_box.set_title(col, fontsize=9, fontweight="bold");
    ax_box.set_xticklabels([]);

    # Row 1 — histogram + KDE
    ax_hist = axes[1, i];
    ax_hist.hist(data, bins=60, color="#4C72B0", alpha=0.7, density=True);
    data.plot.kde(ax=ax_hist, color="#DD8452", linewidth=1.8);
    ax_hist.set_xlabel(col, fontsize=8);
    ax_hist.tick_params(labelsize=7);

axes[0, 0].set_ylabel("Box-plot", fontsize=8);
axes[1, 0].set_ylabel("Histogram + KDE", fontsize=8);
fig.suptitle("Distribution & Outlier Overview (continuous features)",
             fontsize=13, fontweight="bold");
plt.show();

# IQR Summary
print(f"{'Feature':<20} {'Q1':>10} {'Q3':>10} {'IQR':>10} "
      f"{'Lower':>12} {'Upper':>12} {'# Outliers':>12} {'% Outliers':>11}");
print("-" * 100);
for col in NUMERIC_FEATURES:
    data = df[col].dropna();
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75);
    IQR = Q3 - Q1;
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR;
    n_out = ((data < lo) | (data > hi)).sum();
    pct   = n_out / len(data) * 100;
    print(f"{col:<20} {Q1:>10.2f} {Q3:>10.2f} {IQR:>10.2f} "
          f"{lo:>12.2f} {hi:>12.2f} {n_out:>12d} {pct:>10.1f}%");

# Scatter plots between selling_price and other attributes (to catch extreme values)
PREDICTORS = ["km_driven", "mileage_val", "engine_cc", "max_power_bhp", "torque_nm"];
fig, axes = plt.subplots(1, len(PREDICTORS), figsize=(22, 4),
                          gridspec_kw={"wspace": 0.35});
for ax, pred in zip(axes, PREDICTORS):
    ax.scatter(df[pred], df["selling_price"],
               alpha=0.15, s=6, color="#4C72B0");
    ax.set_xlabel(pred, fontsize=8);
    ax.set_ylabel("selling_price", fontsize=8);
    ax.tick_params(labelsize=7);
fig.suptitle("selling_price vs. continuous predictors",
             fontsize=12, fontweight="bold");
plt.show();

# Correlation Heatmap
CORR_COLS = ["selling_price", "km_driven", "mileage_val",
             "engine_cc", "max_power_bhp", "torque_nm", "year"];
fig, ax = plt.subplots(figsize=(8, 6));
sns.heatmap(df[CORR_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 9});
ax.set_title("Pearson Correlation Matrix", fontsize=12, fontweight="bold");
plt.tight_layout();
plt.show();

# LOF Feature Selection
LOF_FEATURES = ["selling_price", "km_driven"];
print("\nLOF applied on features:", LOF_FEATURES);

lof_df = df[LOF_FEATURES].dropna().copy();
print(f"Rows entering LOF: {len(lof_df):,}");

# Scaling based off of median and IQR
X_scaled = RobustScaler().fit_transform(lof_df);

LOF_NEIGHBORS     = 20;
LOF_CONTAMINATION = 0.03;
lof   = LocalOutlierFactor(n_neighbors=LOF_NEIGHBORS, contamination=LOF_CONTAMINATION);
preds  = lof.fit_predict(X_scaled);   # -1 = outlier, +1 = inlier
scores = -lof.negative_outlier_factor_;  # higher = more outlying

lof_df["lof_label"] = preds;
lof_df["lof_score"] = scores;

n_outliers = (preds == -1).sum();
n_inliers  = (preds ==  1).sum();
print(f"LOF results => outliers: {n_outliers:,} | inliers: {n_inliers:,} "
      f"({n_outliers / len(lof_df) * 100:.2f}% flagged)");

# Scatter plot (inliers vs outliers)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"wspace": 0.35});
inl = lof_df[lof_df["lof_label"] ==  1];
out = lof_df[lof_df["lof_label"] == -1];
for ax, log_scale, title_suffix in zip(axes, [False, True], ["raw scale", "log scale"]):
    ax.scatter(inl["km_driven"], inl["selling_price"],
               s=8, alpha=0.25, color="#4C72B0", label=f"Inlier ({n_inliers:,})");
    ax.scatter(out["km_driven"], out["selling_price"],
               s=25, alpha=0.8, color="#E74C3C",
               label=f"LOF outlier ({n_outliers})", zorder=3);
    if log_scale:
        ax.set_xscale("log"); ax.set_yscale("log");
    ax.set_xlabel("km_driven" + (" (log)" if log_scale else ""), fontsize=10);
    ax.set_ylabel("selling_price" + (" (log)" if log_scale else ""), fontsize=10);
    ax.set_title(f"LOF — {title_suffix}", fontsize=11, fontweight="bold");
    ax.legend(fontsize=9);
fig.suptitle("Local Outlier Factor — selling_price × km_driven",
             fontsize=13, fontweight="bold");
plt.show();

# LOF Score Distribution
fig, ax = plt.subplots(figsize=(10, 3));
ax.hist(lof_df[lof_df["lof_label"] ==  1]["lof_score"], bins=80,
        color="#4C72B0", alpha=0.7, label="Inliers");
ax.hist(lof_df[lof_df["lof_label"] == -1]["lof_score"], bins=30,
        color="#E74C3C", alpha=0.85, label="Outliers");
ax.axvline(lof_df[lof_df["lof_label"] == -1]["lof_score"].min(),
           color="black", linestyle="--", linewidth=1.2,
           label="Decision boundary (min outlier score)");
ax.set_xlabel("LOF score (higher = more outlying)", fontsize=10);
ax.set_ylabel("Count", fontsize=10);
ax.set_title("Distribution of LOF scores", fontsize=11, fontweight="bold");
ax.legend(fontsize=9);
plt.tight_layout();
plt.show();

# Top outliers by LOF score
outlier_idx = lof_df[lof_df["lof_label"] == -1].index;
print("\nTop-10 outliers by LOF score:");
print(
    df.loc[outlier_idx, ["name", "year", "selling_price", "km_driven", "fuel"]]
    .assign(lof_score=lof_df.loc[outlier_idx, "lof_score"])
    .sort_values("lof_score", ascending=False)
    .head(10)
    .to_string()
);

# Remove LOF-flagged rows
df_before = df.copy();
df = df.drop(index=outlier_idx).reset_index(drop=True);
print(f"\nRows before removal : {len(df_before):,}");
print(f"Rows removed (LOF)  : {len(outlier_idx):,}");
print(f"Rows after  removal : {len(df):,}");

# Comparison between before and after
fig, axes = plt.subplots(2, 2, figsize=(14, 7),
                          gridspec_kw={"wspace": 0.35, "hspace": 0.45});
for col_i, col in enumerate(["selling_price", "km_driven"]):
    before_data = df_before[col].dropna();
    after_data  = df[col].dropna();

    # Box-plot
    ax_b = axes[0, col_i];
    bp = ax_b.boxplot([before_data, after_data],
                      labels=["Before", "After"],
                      patch_artist=True,
                      medianprops=dict(color="#C0392B", linewidth=2));
    for patch, colour in zip(bp["boxes"], ["#AEC6CF", "#90EE90"]):
        patch.set_facecolor(colour);
    ax_b.set_title(f"{col} — box-plot", fontsize=10, fontweight="bold");

    # Histogram
    ax_h = axes[1, col_i];
    ax_h.hist(before_data, bins=80, alpha=0.55, color="#4C72B0",
              density=True, label="Before");
    ax_h.hist(after_data,  bins=80, alpha=0.55, color="#2ECC71",
              density=True, label="After");
    ax_h.set_xlabel(col, fontsize=9);
    ax_h.set_ylabel("Density", fontsize=9);
    ax_h.set_title(f"{col} — distribution", fontsize=10, fontweight="bold");
    ax_h.legend(fontsize=8);

fig.suptitle("Before vs After LOF Outlier Removal",
             fontsize=13, fontweight="bold");
plt.show();

# Stats comparison
for col in ["selling_price", "km_driven"]:
    print(f"\nDescriptive statistics — {col}:");
    print(pd.DataFrame({
        "Before": df_before[col].describe(),
        "After" : df[col].describe(),
    }).round(2).to_string());

something

In [ ]:
# THIS IS AI - NOT SURE IF OPTIONAL OR NOT

from sklearn.linear_model import LinearRegression;
from sklearn.model_selection import train_test_split, cross_val_score, KFold;
from sklearn.preprocessing import StandardScaler;
from sklearn.pipeline import Pipeline;
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score;

# ── Step 1: Rebuild df_encoded from the post-LOF df ──────────────────────────
# The OHE step earlier in the notebook was applied to the pre-LOF df.
# We rebuild it here from the cleaned df so the design matrix is consistent.
OHE_COLS     = ["fuel", "seller_type", "transmission", "owner"];
NUMERIC_COLS = ["year", "selling_price", "km_driven",
                "mileage_val", "engine_cc", "max_power_bhp", "torque_nm", "seats"];

df_encoded = pd.get_dummies(
    df[NUMERIC_COLS + OHE_COLS],
    columns=OHE_COLS,
    drop_first=True,   # avoids the dummy variable trap
    dtype=int,
);
df_encoded["seats"] = df_encoded["seats"].astype(float);  # Int64 → float64 for sklearn
df_encoded = df_encoded.dropna().reset_index(drop=True);

print(f"Design matrix: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns");
print(f"Features : {df_encoded.shape[1] - 1}");
print(f"Target   : selling_price");
print(f"\nColumn list:\n{df_encoded.columns.tolist()}");

# ── Step 2: Train / test split (80 / 20) ─────────────────────────────────────
TARGET = "selling_price";
X = df_encoded.drop(columns=[TARGET]);
y = df_encoded[TARGET];

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
);
print(f"\nTrain set : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)");
print(f"Test  set : {X_test.shape[0]:,}  rows ({X_test.shape[0]/len(X)*100:.0f}%)");

# ── Step 3: Baseline — unscaled LinearRegression ─────────────────────────────
lr_base = LinearRegression();
lr_base.fit(X_train, y_train);
y_pred_base = lr_base.predict(X_test);

mae_base  = mean_absolute_error(y_test, y_pred_base);
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base));
r2_base   = r2_score(y_test, y_pred_base);

print(f"\n{'='*50}");
print("BASELINE (unscaled LinearRegression)");
print(f"{'='*50}");
print(f"  MAE  : {mae_base:>12,.2f}");
print(f"  RMSE : {rmse_base:>12,.2f}");
print(f"  R²   : {r2_base:>12.4f}");
print(f"  Intercept: {lr_base.intercept_:,.2f}");

# ── Step 4: Scaled pipeline (StandardScaler → LinearRegression) ───────────────
# Scaling does not change predictions but makes coefficients directly comparable.
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr",     LinearRegression()),
]);
lr_pipe.fit(X_train, y_train);
y_pred = lr_pipe.predict(X_test);

mae  = mean_absolute_error(y_test, y_pred);
rmse = np.sqrt(mean_squared_error(y_test, y_pred));
r2   = r2_score(y_test, y_pred);

print(f"\n{'='*50}");
print("SCALED PIPELINE (StandardScaler → LinearRegression)");
print(f"{'='*50}");
print(f"  MAE  : {mae:>12,.2f}");
print(f"  RMSE : {rmse:>12,.2f}");
print(f"  R²   : {r2:>12.4f}");

# ── Step 5: 5-Fold Cross-Validation ──────────────────────────────────────────
cv = KFold(n_splits=5, shuffle=True, random_state=42);
cv_r2   = cross_val_score(lr_pipe, X, y, cv=cv, scoring="r2");
cv_mae  = -cross_val_score(lr_pipe, X, y, cv=cv, scoring="neg_mean_absolute_error");
cv_rmse = np.sqrt(-cross_val_score(lr_pipe, X, y, cv=cv, scoring="neg_mean_squared_error"));

print(f"\n{'='*50}");
print("5-FOLD CROSS-VALIDATION (scaled pipeline)");
print(f"{'='*50}");
print(f"  R²   per fold : {np.round(cv_r2,   4)}");
print(f"  MAE  per fold : {np.round(cv_mae,  0)}");
print(f"  RMSE per fold : {np.round(cv_rmse, 0)}");
print(f"\n  Mean R²   : {cv_r2.mean():.4f}  ±  {cv_r2.std():.4f}");
print(f"  Mean MAE  : {cv_mae.mean():,.0f}  ±  {cv_mae.std():,.0f}");
print(f"  Mean RMSE : {cv_rmse.mean():,.0f}  ±  {cv_rmse.std():,.0f}");

# ── Step 6: Predicted vs Actual + Residuals ───────────────────────────────────
residuals = y_test - y_pred;

fig, axes = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"wspace": 0.35});

# Panel A — predicted vs actual
ax = axes[0];
ax.scatter(y_test, y_pred, alpha=0.25, s=8, color="#4C72B0");
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())];
ax.plot(lims, lims, "--", color="#E74C3C", linewidth=1.5, label="Perfect fit");
ax.set_xlabel("Actual selling_price", fontsize=10);
ax.set_ylabel("Predicted selling_price", fontsize=10);
ax.set_title("A — Predicted vs Actual", fontsize=11, fontweight="bold");
ax.legend(fontsize=9);
ax.annotate(f"R² = {r2:.4f}", xy=(0.05, 0.92), xycoords="axes fraction", fontsize=10,
            color="#333", bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFFACD", alpha=0.8));

# Panel B — residuals vs predicted
ax2 = axes[1];
ax2.scatter(y_pred, residuals, alpha=0.25, s=8, color="#DD8452");
ax2.axhline(0, color="#E74C3C", linewidth=1.5, linestyle="--");
ax2.set_xlabel("Predicted selling_price", fontsize=10);
ax2.set_ylabel("Residual", fontsize=10);
ax2.set_title("B — Residuals vs Predicted", fontsize=11, fontweight="bold");

# Panel C — residual distribution
ax3 = axes[2];
ax3.hist(residuals, bins=60, color="#55A868", alpha=0.75, density=True);
residuals.plot.kde(ax=ax3, color="#333", linewidth=1.8);
ax3.axvline(0, color="#E74C3C", linewidth=1.5, linestyle="--");
ax3.set_xlabel("Residual", fontsize=10);
ax3.set_ylabel("Density", fontsize=10);
ax3.set_title("C — Residual Distribution", fontsize=11, fontweight="bold");

fig.suptitle("Linear Regression — Prediction Diagnostics", fontsize=13, fontweight="bold");
plt.show();

# ── Step 7: Standardised coefficient bar chart ────────────────────────────────
# Coefficients from the scaled pipeline are directly comparable:
# larger absolute value = stronger influence on selling_price.
coef_series = pd.Series(
    lr_pipe.named_steps["lr"].coef_,
    index=X.columns,
).sort_values(key=abs, ascending=True);   # ascending so largest appears at top

colours = ["#E74C3C" if v < 0 else "#4C72B0" for v in coef_series];

fig, ax = plt.subplots(figsize=(10, 7));
bars = ax.barh(coef_series.index, coef_series.values, color=colours, edgecolor="#333", linewidth=0.4);
ax.axvline(0, color="black", linewidth=0.8);
ax.set_xlabel("Standardised coefficient (scaled units)", fontsize=10);
ax.set_title("Feature Coefficients — Scaled LinearRegression",
             fontsize=12, fontweight="bold");

# Annotate bar values
for bar, val in zip(bars, coef_series.values):
    offset = 3000 if val >= 0 else -3000;
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f}", va="center", ha="left" if val >= 0 else "right",
            fontsize=7.5, color="#222");

from matplotlib.patches import Patch;
ax.legend(handles=[
    Patch(facecolor="#4C72B0", label="Positive effect on price"),
    Patch(facecolor="#E74C3C", label="Negative effect on price"),
], fontsize=9, loc="lower right");
plt.tight_layout();
plt.show();

# ── Step 8: Summary table ─────────────────────────────────────────────────────
print(f"\n{'='*55}");
print("RESULTS SUMMARY");
print(f"{'='*55}");
summary_data = {
    "Model":       ["Baseline (unscaled)", "Scaled Pipeline",
                    "5-Fold CV (scaled mean)", "5-Fold CV (scaled std)"],
    "MAE":         [f"{mae_base:,.0f}",  f"{mae:,.0f}",
                    f"{cv_mae.mean():,.0f}",  f"±{cv_mae.std():,.0f}"],
    "RMSE":        [f"{rmse_base:,.0f}", f"{rmse:,.0f}",
                    f"{cv_rmse.mean():,.0f}", f"±{cv_rmse.std():,.0f}"],
    "R²":          [f"{r2_base:.4f}",   f"{r2:.4f}",
                    f"{cv_r2.mean():.4f}",   f"±{cv_r2.std():.4f}"],
};
print(pd.DataFrame(summary_data).to_string(index=False));

print(f"\nTop 5 features by absolute standardised coefficient:");
print(coef_series.abs().sort_values(ascending=False).head(5).round(0).to_string());

Something the sequel

In [ ]:
CURRENT_YEAR = 2026;

# Feature 1: power_to_displacement (specific power — bhp per CC)
# Captures engine performance efficiency independently of raw engine size.
# Both source columns are already imputed, so NaNs here are negligible.
df["power_to_displacement"] = df["max_power_bhp"] / df["engine_cc"];

# Feature 2: km_per_year (annual usage intensity — km / year)
# Normalises the odometer by the car's age so usage intensity is comparable
# across cars of different vintages. Age is floored at 1 to avoid division by zero.
df["car_age"]     = (CURRENT_YEAR - df["year"]).clip(lower=1);
df["km_per_year"] = df["km_driven"] / df["car_age"];

# Sanity Check
NEW_FEATURES = ["power_to_displacement", "km_per_year"];

print("Aggregated feature summary:");
print(df[NEW_FEATURES].describe().round(4).to_string());
print();
for feat in NEW_FEATURES:
    corr = df[feat].corr(df["selling_price"]);
    nans = df[feat].isna().sum();
    print(f"  {feat:<25}  NaN: {nans:>3}   Corr w/ selling_price: {corr:+.4f}");

# Visual Aids
fig, axes = plt.subplots(2, 3, figsize=(18, 9),
                          gridspec_kw={"hspace": 0.45, "wspace": 0.35});

for row, feat in enumerate(NEW_FEATURES):
    data = df[feat].dropna();

    # Column 0: histogram + KDE
    ax0 = axes[row, 0];
    ax0.hist(data, bins=60, color="#4C72B0", alpha=0.7, density=True);
    data.plot.kde(ax=ax0, color="#DD8452", linewidth=1.8);
    ax0.set_title(f"{feat}\nDistribution", fontsize=10, fontweight="bold");
    ax0.set_xlabel(feat, fontsize=8);
    ax0.set_ylabel("Density", fontsize=8);

    # Column 1: box-plot
    ax1 = axes[row, 1];
    ax1.boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor="#AEC6CF", color="#333"),
                medianprops=dict(color="#C0392B", linewidth=2),
                flierprops=dict(marker="o", markerfacecolor="#E74C3C",
                                markersize=3, alpha=0.5));
    ax1.set_title(f"{feat}\nBox-plot", fontsize=10, fontweight="bold");
    ax1.set_xticklabels([]);

    # Column 2: scatter vs selling_price with trend line
    ax2 = axes[row, 2];
    ax2.scatter(df[feat], df["selling_price"],
                alpha=0.15, s=6, color="#55A868");
    mask = df[feat].notna() & df["selling_price"].notna();
    z = np.polyfit(df.loc[mask, feat], df.loc[mask, "selling_price"], 1);
    xline = np.linspace(data.min(), data.max(), 200);
    ax2.plot(xline, np.polyval(z, xline), color="#E74C3C",
             linewidth=1.8, label="Linear trend");
    corr_val = df[feat].corr(df["selling_price"]);
    ax2.set_title(f"{feat} vs selling_price\nr = {corr_val:+.4f}",
                  fontsize=10, fontweight="bold");
    ax2.set_xlabel(feat, fontsize=8);
    ax2.set_ylabel("selling_price", fontsize=8);
    ax2.legend(fontsize=8);
    ax2.tick_params(labelsize=7);

fig.suptitle("Aggregated Features — Distribution & Relationship with Target",
             fontsize=13, fontweight="bold");
plt.show();

# Correlation Comparison: Raw inputs vs aggregated
comp_cols = {
    "max_power_bhp"         : "raw input A",
    "engine_cc"             : "raw input B",
    "power_to_displacement" : "aggregated  (A / B)",
    "km_driven"             : "raw input C",
    "year"                  : "raw input D  (age proxy)",
    "km_per_year"           : "aggregated  (C / age)",
};
print("\nPearson correlation with selling_price:");
print(f"  {'Feature':<28} {'Role':<28} {'r':>8}");
print("  " + "-" * 68);
for col, role in comp_cols.items():
    r = df[col].corr(df["selling_price"]);
    print(f"  {col:<28} {role:<28} {r:>+8.4f}");

Something again

In [ ]:
from sklearn.linear_model import LinearRegression;
from sklearn.model_selection import train_test_split, KFold, cross_val_score;
from sklearn.preprocessing import StandardScaler;
from sklearn.pipeline import Pipeline;
from sklearn.metrics import mean_squared_error;

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Carve out the 15% test set FIRST
# This happens before any LOF removal or feature aggregation so that the test
# set is completely invisible to all configuration-specific preprocessing steps.
# ══════════════════════════════════════════════════════════════════════════════
TARGET       = "selling_price";
OHE_COLS     = ["fuel", "seller_type", "transmission", "owner"];
NUMERIC_COLS = ["year", "selling_price", "km_driven",
                "mileage_val", "engine_cc", "max_power_bhp", "torque_nm", "seats"];
CURRENT_YEAR = 2026;
RANDOM_STATE = 42;

df_trainval, df_test = train_test_split(df, test_size=0.15, random_state=RANDOM_STATE);
df_trainval = df_trainval.reset_index(drop=True);
df_test     = df_test.reset_index(drop=True);

print("Dataset split:");
print(f"  Train + Validation : {len(df_trainval):,} rows ({len(df_trainval)/len(df)*100:.1f}%)");
print(f"  Test (locked)      : {len(df_test):,}  rows ({len(df_test)/len(df)*100:.1f}%)");

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Helper functions
# ══════════════════════════════════════════════════════════════════════════════
def apply_lof(df_in):
    """
    Fit LOF on df_in and return only the inlier rows.
    LOF is always fitted exclusively on the data passed in — never on
    validation or test rows.
    """
    lof_data = df_in[["selling_price", "km_driven"]].copy();
    X_sc = RobustScaler().fit_transform(lof_data);
    lof  = LocalOutlierFactor(n_neighbors=20, contamination=0.03);
    mask = lof.fit_predict(X_sc) == 1;  # True = inlier
    return df_in[mask].reset_index(drop=True);

def build_design_matrix(df_in, use_feat_agg=False):
    """
    One-hot encode categorical columns and (optionally) add the two
    aggregated features. Returns the fully numeric design matrix.
    """
    d = df_in.copy();
    num_cols = NUMERIC_COLS.copy();
    if use_feat_agg:
        d["power_to_displacement"] = d["max_power_bhp"] / d["engine_cc"];
        d["car_age"]     = (CURRENT_YEAR - d["year"]).clip(lower=1);
        d["km_per_year"] = d["km_driven"] / d["car_age"];
        num_cols += ["power_to_displacement", "km_per_year"];
    enc = pd.get_dummies(d[num_cols + OHE_COLS],
                         columns=OHE_COLS, drop_first=True, dtype=int);
    enc["seats"] = enc["seats"].astype(float);
    return enc.dropna().reset_index(drop=True);

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — 4-fold cross-validation for each configuration
# ══════════════════════════════════════════════════════════════════════════════
CONFIGS = {
    "Baseline  (no LOF, no agg)" : {"use_lof": False, "use_agg": False},
    "Config 1  (LOF,    no agg)" : {"use_lof": True,  "use_agg": False},
    "Config 2  (no LOF, agg)"    : {"use_lof": False, "use_agg": True},
    "Config 3  (LOF  +  agg)"    : {"use_lof": True,  "use_agg": True},
};

cv_results = {};

print("\nRunning 4-fold cross-validation on Train+Validation set...");
print(f"{'Configuration':<35} {'Mean CV MSE':>22} {'Std CV MSE':>18}");
print("-" * 78);

for cfg_name, cfg in CONFIGS.items():
    # Apply LOF only to the training data, if requested
    d = apply_lof(df_trainval) if cfg["use_lof"] else df_trainval.copy();
    enc  = build_design_matrix(d, use_feat_agg=cfg["use_agg"]);
    X    = enc.drop(columns=[TARGET]);
    y    = enc[TARGET];
    pipe = Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())]);
    kf   = KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE);
    fold_mses = -cross_val_score(pipe, X, y, cv=kf,
                                  scoring="neg_mean_squared_error");
    cv_results[cfg_name] = {
        "fold_mses" : fold_mses,
        "mean_mse"  : fold_mses.mean(),
        "std_mse"   : fold_mses.std(),
        "use_lof"   : cfg["use_lof"],
        "use_agg"   : cfg["use_agg"],
    };
    print(f"{cfg_name:<35} {fold_mses.mean():>22,.0f} {fold_mses.std():>18,.0f}");

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Visualise CV MSE per configuration
# ══════════════════════════════════════════════════════════════════════════════
cfg_names  = list(cv_results.keys());
means      = [cv_results[n]["mean_mse"]  for n in cfg_names];
stds       = [cv_results[n]["std_mse"]   for n in cfg_names];
fold_data  = [cv_results[n]["fold_mses"] for n in cfg_names];

COLOURS = ["#95a5a6", "#3498db", "#2ecc71", "#e74c3c"];  # grey, blue, green, red

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.4});

# Panel A — mean CV MSE bar chart with std error bars
ax = axes[0];
bars = ax.bar(cfg_names, means, color=COLOURS, edgecolor="#333",
              linewidth=0.6, zorder=3);
ax.errorbar(cfg_names, means, yerr=stds, fmt="none",
            color="black", capsize=6, linewidth=1.5, zorder=4);
ax.set_ylabel("Mean CV MSE", fontsize=10);
ax.set_title("A — Mean 4-Fold CV MSE per Configuration",
             fontsize=11, fontweight="bold");
ax.tick_params(axis="x", labelsize=8);
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
            f"{val/1e9:.2f}B", ha="center", va="bottom", fontsize=8);
ax.set_yscale("log");
ax.grid(axis="y", alpha=0.3, zorder=0);

# Panel B — per-fold MSE scatter to show stability across folds
ax2 = axes[1];
for i, (name, folds, colour) in enumerate(
        zip(cfg_names, fold_data, COLOURS)):
    ax2.scatter([i] * 4, folds, color=colour, s=60, zorder=3,
                label=name.strip());
    ax2.plot([i - 0.25, i + 0.25], [folds.mean(), folds.mean()],
             color=colour, linewidth=2.5, zorder=4);
ax2.set_xticks(range(len(cfg_names)));
ax2.set_xticklabels([f"Cfg {i}" if i > 0 else "Base"
                     for i in range(len(cfg_names))], fontsize=9);
ax2.set_ylabel("MSE per fold", fontsize=10);
ax2.set_title("B — Per-Fold MSE Stability", fontsize=11, fontweight="bold");
ax2.legend(fontsize=7, loc="upper right");
ax2.grid(axis="y", alpha=0.3);

fig.suptitle("4-Fold Cross-Validation MSE — Configuration Comparison",
             fontsize=13, fontweight="bold");
plt.show();

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Select best configuration
# ══════════════════════════════════════════════════════════════════════════════
best_name = min(cv_results, key=lambda k: cv_results[k]["mean_mse"]);
best_cfg  = cv_results[best_name];

print(f"\n{'='*60}");
print("BEST CONFIGURATION (lowest mean CV MSE)");
print(f"{'='*60}");
print(f"  Name             : {best_name}");
print(f"  LOF removal      : {best_cfg['use_lof']}");
print(f"  Feature agg      : {best_cfg['use_agg']}");
print(f"  Mean CV MSE      : {best_cfg['mean_mse']:,.0f}");
print(f"  Std  CV MSE      : {best_cfg['std_mse']:,.0f}");
print(f"  CV MSE per fold  : {[f'{v:,.0f}' for v in best_cfg['fold_mses']]}");

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — Retrain best config on full Train+Validation, evaluate on Test
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}");
print("FINAL EVALUATION ON HELD-OUT TEST SET");
print(f"{'='*60}");

# Prepare full train+val set under the best configuration
d_full = apply_lof(df_trainval) if best_cfg["use_lof"] else df_trainval.copy();
enc_full = build_design_matrix(d_full,    use_feat_agg=best_cfg["use_agg"]);
enc_test = build_design_matrix(df_test,   use_feat_agg=best_cfg["use_agg"]);

# Align test columns to training columns (handles any OHE category mismatch)
enc_test = enc_test.reindex(columns=enc_full.columns, fill_value=0);

X_full = enc_full.drop(columns=[TARGET]);  y_full = enc_full[TARGET];
X_test = enc_test.drop(columns=[TARGET]);  y_test = enc_test[TARGET];

print(f"  Train+Val rows used for final fit : {len(X_full):,}");
print(f"  Test rows for final evaluation    : {len(X_test):,}");

final_pipe = Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())]);
final_pipe.fit(X_full, y_full);
y_pred_test = final_pipe.predict(X_test);

test_mse  = mean_squared_error(y_test, y_pred_test);
test_rmse = np.sqrt(test_mse);
test_mae  = np.mean(np.abs(y_test - y_pred_test));
test_r2   = 1 - (np.sum((y_test - y_pred_test)**2) /
                 np.sum((y_test - y_test.mean())**2));

print(f"\n  Final Test MSE  : {test_mse:>20,.2f}");
print(f"  Final Test RMSE : {test_rmse:>20,.2f}");
print(f"  Final Test MAE  : {test_mae:>20,.2f}");
print(f"  Final Test R²   : {test_r2:>20.4f}");

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — Final diagnostic plots
# ══════════════════════════════════════════════════════════════════════════════
residuals = y_test - y_pred_test;

fig, axes = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"wspace": 0.35});

# Panel A — Predicted vs Actual
ax = axes[0];
ax.scatter(y_test, y_pred_test, alpha=0.25, s=8, color="#4C72B0");
lims = [min(y_test.min(), y_pred_test.min()),
        max(y_test.max(), y_pred_test.max())];
ax.plot(lims, lims, "--", color="#E74C3C", linewidth=1.5, label="Perfect fit");
ax.set_xlabel("Actual selling_price", fontsize=10);
ax.set_ylabel("Predicted selling_price", fontsize=10);
ax.set_title("A — Predicted vs Actual", fontsize=11, fontweight="bold");
ax.legend(fontsize=9);
ax.annotate(f"R² = {test_r2:.4f}",
            xy=(0.05, 0.92), xycoords="axes fraction", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFFACD", alpha=0.8));

# Panel B — Residuals vs Predicted
ax2 = axes[1];
ax2.scatter(y_pred_test, residuals, alpha=0.25, s=8, color="#DD8452");
ax2.axhline(0, color="#E74C3C", linewidth=1.5, linestyle="--");
ax2.set_xlabel("Predicted selling_price", fontsize=10);
ax2.set_ylabel("Residual", fontsize=10);
ax2.set_title("B — Residuals vs Predicted", fontsize=11, fontweight="bold");

# Panel C — Residual Distribution
ax3 = axes[2];
pd.Series(residuals).hist(ax=ax3, bins=60, color="#55A868", alpha=0.75, density=True);
pd.Series(residuals).plot.kde(ax=ax3, color="#333", linewidth=1.8);
ax3.axvline(0, color="#E74C3C", linewidth=1.5, linestyle="--");
ax3.set_xlabel("Residual", fontsize=10);
ax3.set_ylabel("Density", fontsize=10);
ax3.set_title("C — Residual Distribution", fontsize=11, fontweight="bold");

fig.suptitle(
    f"Final Test Evaluation — Best Config: {best_name.strip()}",
    fontsize=13, fontweight="bold"
);
plt.show();

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — Summary table
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*72}");
print("FULL RESULTS SUMMARY");
print(f"{'='*72}");
print(f"{'Configuration':<35} {'Mean CV MSE':>20} {'Std':>15} {'Selected':>10}");
print("-" * 82);
for name, res in cv_results.items():
    marker = "  ◄ BEST" if name == best_name else "";
    print(f"{name:<35} {res['mean_mse']:>20,.0f} {res['std_mse']:>15,.0f}{marker}");
print("-" * 82);
print(f"{'Final Test MSE  (best config)':<35} {test_mse:>20,.0f}");
print(f"{'Final Test RMSE (best config)':<35} {test_rmse:>20,.0f}");